# QuantAlpha AI — Step 2B: Event Classifier (Fixed + Robust)
The previous version had a bug: yfinance changed its news data structure, so headlines came back
empty and every stock got an identical, meaningless classification.

This version:
1. **Diagnoses** the raw news structure first, so we see exactly what's coming back
2. **Handles multiple yfinance formats** (old and new)
3. **Falls back to Google News RSS** (free, no API key) for any stock where yfinance news is
   empty — since Yahoo's news coverage for Indian small/mid-caps is often thin anyway
4. Re-runs FinBERT + zero-shot classification properly on real headlines


## 1. Connect to Drive + install

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/QuantAlpha')
print("Working directory:", os.getcwd())


Mounted at /content/drive
Working directory: /content/drive/MyDrive/QuantAlpha


In [2]:
!pip install transformers torch yfinance feedparser --quiet
print("Dependencies ready.")


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 420.5 kB/s eta 0:00:00
Dependencies ready.


In [3]:
import pandas as pd
import numpy as np
import yfinance as yf
import feedparser
import urllib.parse
import warnings
warnings.filterwarnings('ignore')


## 2. Diagnostic — see what yfinance actually returns
Run this first. It prints the raw structure of one news item so we know exactly which keys
hold the headline, instead of guessing.


In [4]:
test_symbol = "IDFCFIRSTB.NS"
t = yf.Ticker(test_symbol)
raw_news = t.news

print(f"Number of news items returned: {len(raw_news) if raw_news else 0}")
if raw_news:
    print("\nRaw structure of first item:")
    import json
    print(json.dumps(raw_news[0], indent=2, default=str)[:2000])
else:
    print("yfinance returned NO news for this symbol — this is common for Indian stocks.")
    print("We'll rely on the Google News RSS fallback instead (next cells).")


Number of news items returned: 10

Raw structure of first item:
{
  "id": "6e6e97ba-b94b-3026-8a0a-3480da000720",
  "content": {
    "id": "6e6e97ba-b94b-3026-8a0a-3480da000720",
    "contentType": "STORY",
    "title": "IDFC First Bank Ltd (BOM:539437) Q4 2026 Earnings Call Highlights: Strong Loan Growth Amidst ...",
    "description": "",
    "summary": "IDFC First Bank Ltd (BOM:539437) reports robust loan growth and improved asset quality, despite facing a one-off fraud incident and modest deposit growth.",
    "pubDate": "2026-04-27T07:00:41Z",
    "displayTime": "2026-04-27T07:00:41Z",
    "isHosted": true,
    "bypassModal": false,
    "previewUrl": null,
    "thumbnail": null,
    "provider": {
      "displayName": "GuruFocus.com",
      "url": "http://www.gurufocus.com/",
      "sourceId": "us.finance.gurufocus"
    },
    "canonicalUrl": {
      "url": "https://finance.yahoo.com/markets/stocks/articles/idfc-first-bank-ltd-bom-070041097.html",
      "site": "finance",
      "re

**Read the printed structure above.** Depending on what you see, yfinance news is either:
- Empty (`[]`) → very common for Indian tickers, Yahoo's news coverage is thin outside US stocks
- Present but nested differently than expected → our extractor below handles the known variants

Either way, the RSS fallback below is the more reliable path for Indian stocks specifically.


## 3. Google News RSS fallback (free, no API key, reliable for Indian stocks)
Google News publishes a free RSS feed you can query by search term — no key, no rate-limit
headaches, and much better coverage for Indian company news than Yahoo's news API.


In [5]:
def fetch_news_rss(company_name, max_items=5):
    """Fetch recent news headlines via Google News RSS search — free, no API key."""
    query = urllib.parse.quote(f"{company_name} stock India")
    url = f"https://news.google.com/rss/search?q={query}&hl=en-IN&gl=IN&ceid=IN:en"
    try:
        feed = feedparser.parse(url)
        items = []
        for entry in feed.entries[:max_items]:
            items.append({
                'title': entry.get('title', ''),
                'published': entry.get('published', ''),
                'source': entry.get('source', {}).get('title', '') if hasattr(entry.get('source', {}), 'get') else ''
            })
        return items
    except Exception as e:
        print(f"RSS fetch failed for {company_name}: {e}")
        return []

# Quick test
test_news = fetch_news_rss("IDFC First Bank")
print(f"Fetched {len(test_news)} headlines via RSS")
for n in test_news:
    print(" -", n['title'])


Fetched 5 headlines via RSS
 - IDFC First Bank Ltd up for third straight session - Business Standard
 - IDFC First Bank Q1 update: Loan book grows 21% YoY to Rs 3.05 lakh crore; shares rise 2% - The Economic Times
 - IDFC First Bank share price falls over 2% amid weak sentiment in Indian stock market - livemint.com
 - YES Bank, HDFC Bank, Axis Bank, RBL Bank, IDFC First: Banking stocks to buy ahead of Q1 show - Business Today
 - IDFC First Bank Q1 loans up 21%, CASA ratio tops 50% - MSN


## 4. Robust headline extractor (handles yfinance old/new formats + RSS)

In [6]:
def extract_headline_yf(item):
    """Handles multiple yfinance news structures seen across versions."""
    if not isinstance(item, dict):
        return ''
    # New format: nested under 'content'
    if 'content' in item and isinstance(item['content'], dict):
        title = item['content'].get('title', '')
        if title:
            return title
        return item['content'].get('summary', '')
    # Old format: direct 'title' key
    if 'title' in item and item['title']:
        return item['title']
    return ''

def get_news_for_stock(symbol, company_name, max_items=5):
    """Try yfinance first, fall back to RSS if empty or headlines extract blank."""
    headlines = []
    try:
        t = yf.Ticker(symbol)
        raw = t.news
        for item in (raw or [])[:max_items]:
            h = extract_headline_yf(item)
            if h:
                headlines.append(h)
    except Exception:
        pass

    if not headlines:
        rss_items = fetch_news_rss(company_name, max_items=max_items)
        headlines = [item['title'] for item in rss_items if item['title']]

    return headlines


## 5. Load models (FinBERT + zero-shot event classifier)

In [7]:
from transformers import pipeline

print("Loading FinBERT (financial sentiment)...")
finbert = pipeline("sentiment-analysis", model="ProsusAI/finbert")

print("Loading zero-shot event classifier (BART-MNLI)...")
zero_shot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

EVENT_CATEGORIES = [
    "one-time isolated incident (theft, accident, fraud by an individual)",
    "regulatory or compliance issue",
    "leadership or management change",
    "earnings miss or guidance cut",
    "structural business decline",
    "macroeconomic or sector-wide event",
    "routine business update with no major impact"
]
print("Models loaded.")


Loading FinBERT (financial sentiment)...


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Models loaded.


In [8]:
def classify_event(headline):
    text = headline.strip()
    if not text:
        return None
    sentiment = finbert(text[:512])[0]
    event = zero_shot(text[:512], EVENT_CATEGORIES)
    top_category = event['labels'][0]
    top_score = event['scores'][0]

    temporary_categories = [
        "one-time isolated incident (theft, accident, fraud by an individual)",
        "routine business update with no major impact"
    ]
    permanent_categories = [
        "structural business decline",
        "earnings miss or guidance cut",
        "regulatory or compliance issue"
    ]
    if top_category in temporary_categories:
        verdict = "LIKELY TEMPORARY — dip may be a buying opportunity if fundamentals are stable"
    elif top_category in permanent_categories:
        verdict = "POSSIBLE STRUCTURAL ISSUE — check fundamentals carefully before buying the dip"
    else:
        verdict = "UNCLEAR — needs fundamental cross-check"

    return {
        'headline': headline,
        'sentiment': sentiment['label'],
        'sentiment_confidence': round(sentiment['score'], 3),
        'event_category': top_category,
        'category_confidence': round(top_score, 3),
        'verdict': verdict
    }


## 6. Test on IDFC First Bank (your reference example)

In [9]:
headlines = get_news_for_stock("IDFCFIRSTB.NS", "IDFC First Bank", max_items=5)
print(f"Got {len(headlines)} real headlines\n")

for h in headlines:
    result = classify_event(h)
    if result:
        print(f"Headline: {result['headline']}")
        print(f"  Sentiment: {result['sentiment']} ({result['sentiment_confidence']})")
        print(f"  Event type: {result['event_category']} ({result['category_confidence']})")
        print(f"  Verdict: {result['verdict']}\n")


Got 5 real headlines

Headline: IDFC First Bank Ltd (BOM:539437) Q4 2026 Earnings Call Highlights: Strong Loan Growth Amidst ...
  Sentiment: positive (0.955)
  Event type: macroeconomic or sector-wide event (0.422)
  Verdict: UNCLEAR — needs fundamental cross-check

Headline: India's IDFC First Bank pays 6.45 billion rupees to settle claims tied to suspected fraud
  Sentiment: neutral (0.493)
  Event type: macroeconomic or sector-wide event (0.34)
  Verdict: UNCLEAR — needs fundamental cross-check

Headline: Indian state chief minister says IDFC First Bank delayed acting on suspected fraud
  Sentiment: negative (0.682)
  Event type: leadership or management change (0.323)
  Verdict: UNCLEAR — needs fundamental cross-check

Headline: India's IDFC First Bank tanks 20% on suspected $65 million fraud
  Sentiment: positive (0.884)
  Event type: macroeconomic or sector-wide event (0.412)
  Verdict: UNCLEAR — needs fundamental cross-check

Headline: Trending tickers: Rolls-Royce, Fresnillo, 

**Check this output carefully before proceeding.** You should now see real, different
headlines with varying sentiment/category — not identical blank results like before.


## 7. Run across all Nifty 200 stocks
Uses your Step 1 stock list. We need company names for the RSS fallback — pulled from
yfinance's `.info['longName']` where available, otherwise falls back to just the symbol.


In [10]:
import glob

ohlcv_files = glob.glob('data/ohlcv/*.parquet')
symbols = sorted(set(f.split('/')[-1].replace('.parquet', '') + '.NS' for f in ohlcv_files))
print(f"Processing {len(symbols)} stocks...")

def get_company_name(symbol):
    try:
        info = yf.Ticker(symbol).info
        return info.get('longName', info.get('shortName', symbol))
    except Exception:
        return symbol

all_event_results = []
processed = 0

for sym in symbols:
    name = get_company_name(sym)
    headlines = get_news_for_stock(sym, name, max_items=3)
    for h in headlines:
        result = classify_event(h)
        if result and result['sentiment'] == 'negative':
            result['symbol'] = sym
            all_event_results.append(result)
    processed += 1
    if processed % 25 == 0:
        print(f"Progress: {processed}/{len(symbols)} | Negative events found so far: {len(all_event_results)}")

events_df = pd.DataFrame(all_event_results)
print(f"\nDone. Found {len(events_df)} negative-sentiment news items across the universe")
events_df.to_csv('data/event_classification.csv', index=False)
events_df.head(15)


Processing 200 stocks...


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Progress: 25/200 | Negative events found so far: 7
Progress: 50/200 | Negative events found so far: 20
Progress: 75/200 | Negative events found so far: 29
Progress: 100/200 | Negative events found so far: 45
Progress: 125/200 | Negative events found so far: 54
Progress: 150/200 | Negative events found so far: 64
Progress: 175/200 | Negative events found so far: 73
Progress: 200/200 | Negative events found so far: 92

Done. Found 92 negative-sentiment news items across the universe


,headline,sentiment,sentiment_confidence,event_category,category_confidence,verdict,symbol
0,Engineering firm ABB India's profit falls as o...,negative,0.970,earnings miss or guidance cut,0.485,POSSIBLE STRUCTURAL ISSUE — check fundamentals...,ABB.NS
1,US prosecutors drop fraud charges against bill...,negative,0.811,leadership or management change,0.400,UNCLEAR — needs fundamental cross-check,ADANIGREEN.NS
2,Ashok Leyland (NSEI:ASHOKLEY) Stock Fair Value...,negative,0.962,macroeconomic or sector-wide event,0.285,UNCLEAR — needs fundamental cross-check,ASHOKLEY.NS
3,India's Asian Paints beats quarterly profit vi...,negative,0.620,macroeconomic or sector-wide event,0.376,UNCLEAR — needs fundamental cross-check,ASIANPAINT.NS
4,Some lenders hike rates on FX deposits for non...,negative,0.548,regulatory or compliance issue,0.337,POSSIBLE STRUCTURAL ISSUE — check fundamentals...,AUBANK.NS
5,India's Axis Bank CFO resigns,negative,0.829,leadership or management change,0.904,UNCLEAR — needs fundamental cross-check,AXISBANK.NS
6,Indian non-bank lenders plan $1.6 billion in d...,negative,0.955,macroeconomic or sector-wide event,0.431,UNCLEAR — needs fundamental cross-check,BAJFINANCE.NS
7,Bharat Dynamics shares tank 8% after weak Q4 r...,negative,0.941,earnings miss or guidance cut,0.776,POSSIBLE STRUCTURAL ISSUE — check fundamentals...,BDL.NS
8,India's Cyient misses quarterly profit view on...,negative,0.954,earnings miss or guidance cut,0.894,POSSIBLE STRUCTURAL ISSUE — check fundamentals...,BEL.NS
9,Indo-Pacific Defence: the top-performing defen...,negative,0.952,macroeconomic or sector-wide event,0.447,UNCLEAR — needs fundamental cross-check,BEL.NS


## 8. Summary
`data/event_classification.csv` now contains real, distinct headlines with sentiment and event
classification — ready to cross-check against `data/fundamental_scores.csv` from Step 2 Part A.

**Reminder:** treat the "verdict" as a first-pass filter, not a final answer. Always check the
Piotroski F-Score / ROCE trend for that stock before deciding a negative-news dip is a buying
opportunity.

**Next: Step 3** — combine technical indicators (Step 1) + fundamental scores (Step 2 Part A) +
event classification (this notebook) into the mode-aware ranking engine (Swing / Position /
Long-Term), then backtest.
